# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded:", df.shape)

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Dataset loaded: (30000, 44)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = df.copy()

# In this dataset, zero means no position data
if "avg_position" in data.columns:
    data["avg_position"] = data["avg_position"].replace(
        0,
        np.nan
    )

numeric_candidates = [
    "clicks",
    "impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "word_count"
]

categorical_candidates = [
    "content_type"
]

numeric_features = [
    column for column in numeric_candidates
    if column in data.columns
]

categorical_features = [
    column for column in categorical_candidates
    if column in data.columns
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count']
Categorical features: ['content_type']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
missing_flag_columns = []

for column in numeric_features:
    if data[column].isna().any():
        flag_name = f"has_{column}"

        data[flag_name] = (
            data[column].notna().astype(int)
        )

        missing_flag_columns.append(flag_name)

numeric_features = (
    numeric_features + missing_flag_columns
)

print("Missing-value flags:", missing_flag_columns)

Missing-value flags: ['has_avg_position', 'has_scroll_rate', 'has_word_count']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
selected_features = (
    numeric_features + categorical_features
)

# 'is_declining_label' is derived from the 'trend_direction' column
# where 'down' indicates a declining trend.
data["is_declining_label"] = (data["trend_direction"] == "down").astype(int)

X_raw = data[selected_features].copy()

target_column = "is_declining_label"

if target_column not in data.columns:
    raise ValueError(
        f"{target_column} was not found in the CSV."
    )

y = data[target_column].copy()

print("X shape before encoding:", X_raw.shape)
print("y shape:", y.shape)
print("\nTarget counts:")
print(y.value_counts(dropna=False))

Columns in data DataFrame: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'has_avg_position', 'has_scroll_rate', 'has_word_count']
X shape before encoding: (30000, 10)
y shape: (30000,)

Target counts:
is_declining_label
0    30000
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.